## Step 0: Load the Python packages


In [ ]:
# --- Standard Python tools (come with Python) ---
from pathlib import Path        # build and check file/folder paths

# --- Scientific Python tools ---
import numpy as np              # numbers and arrays (lists of numbers you can do math on)
import pandas as pd             # tables
import matplotlib.pyplot as plt # plotting
from IPython.display import display   # shows a table nicely in the notebook

# --- MNE-Python: the MEG analysis package ---
import mne

# --- MNE-BIDS: reads BIDS files together with their sidecar files (Step 2) ---
import mne_bids

# Only show MNE's warnings and errors (not every info message)
mne.set_log_level("WARNING")

# Draw plots inside the notebook
%matplotlib inline

print("MNE version:", mne.__version__)
print("MNE-BIDS version:", mne_bids.__version__)

## Step 2: Find the file and load the filtered and downsampled data

This cell works out the path of the synced MEG file from your settings, checks it exists, and reads it.
If something is wrong it stops with a message that says what to fix.

The filtered and downsampled data (see Notebook 2) are stored in you /derivatives/filtered/ folder.


In [ ]:
# ---------------------------------------------------------------------------
# WHICH RECORDING
# ---------------------------------------------------------------------------
BIDS_ROOT = Path("/jukebox/[yourpath]/auditory-oddball/bids")  # top folder of your BIDS dataset
SUBJECT = "01"           # the part after "sub-"  (as text, in quotes)
SESSION = "01"           # the part after "ses-"; use None (no quotes) if you have no sessions
TASK = "oddballftcued"   # the part after "task-"
RUN = None               # the part after "run-", e.g. "01"; use None if there is no run label
FILTERED_DERIV = "filtered"  # the derivatives folder this notebook saves the filtered, downsampled recording to (Step 4)

# ---------------------------------------------------------------------------
# SAVING (Step 4)
# ---------------------------------------------------------------------------
OVERWRITE = True             # True: replace the saved recording from an earlier run of this notebook

BIDS_ROOT = Path(BIDS_ROOT)

if "[yourpath]" in str(BIDS_ROOT):
    raise ValueError("BIDS_ROOT still contains the [yourpath] placeholder. Set it in Step 1.")

# ---- The BIDS-style name of this recording, e.g. sub-01_ses-01_task-oddball ------------------
sub_ses = [f"sub-{SUBJECT}"] + ([f"ses-{SESSION}"] if SESSION else [])   # folder levels
name_parts = sub_ses + [f"task-{TASK}"] + ([f"run-{RUN}"] if RUN else [])
base = "_".join(name_parts)        # the start of every file name for this recording
sub_dir = Path(*sub_ses)           # e.g. sub-01/ses-01

# ---- The synced MEG file from preprocessingnotebook 02  --------------------------------
filtered_root = BIDS_ROOT / "derivatives" / FILTERED_DERIV
# BIDSPath describes a file by its BIDS labels.
bids_path = mne_bids.BIDSPath(root=filtered_root, subject=SUBJECT, session=SESSION, task=TASK, run=RUN,
                              processing="filtered", datatype="meg", suffix="meg", extension=".fif")
if not bids_path.fpath.is_file() and bids_path.copy().update(split="01").fpath.is_file():
    bids_path.update(split="01")
meg_file = bids_path.fpath
meg_dir = meg_file.parent

if not meg_file.is_file():
    found = sorted(meg_dir.glob("*_meg.fif")) if meg_dir.is_dir() else []
    raise FileNotFoundError(f"Synced MEG file not found:\n    {meg_file}\nHas notebook 02 been run for this "
                            "session? FIF files in that folder: " + (", ".join(p.name for p in found) or "none"))

# ---- The synced recording's _channels.tsv (written by notebook 02, next to the FIF file) -----------
# It lists every channel, the eye-tracking ones included, with any channels already marked bad and why.
# Notebook 01 saved an updated copy, with its bad channels, in derivatives/sensorqc/ (used in Step 2c).
bids_channels_tsv = bids_path.copy().update(split=None, suffix="channels", extension=".tsv").fpath
if not bids_channels_tsv.is_file():
    raise FileNotFoundError(f"The synced recording's _channels.tsv was not found:\n    {bids_channels_tsv}\n"
                            "Re-run notebook 02 for this session; newer versions write it.")

# ---- Read it --------------------------------------------------------------------------------
# read_raw_bids reads the FIF file together with its sidecar files in derivatives/filtered/: the channel types
# and bad channels come from _channels.tsv, and the task and eye-tracking events from _events.tsv (they appear
# as annotations).
#
# You will see one warning here, "participants.tsv file not found". It is expected and harmless:
# read_raw_bids also looks for participants.tsv (age, sex, handedness, ...) in the top folder of the dataset
# it reads from, to copy those details into the recording's header. Here that folder is derivatives/filtered/,
# but participants.tsv only exists at the top of the main BIDS dataset, so it isn't found and those details
# are not added. Nothing in this notebook uses them.
raw = mne_bids.read_raw_bids(bids_path)
raw.load_data()   # read the data themselves into memory (the steps below need them)

print("Loaded:", meg_file)

if len(raw.filenames) > 1:
    print(f"(the recording is split into {len(raw.filenames)} files; MNE read them all)")

# read_raw_bids loses the coil type of the eye-tracker channels (see notebook 01, Step 2b);
# the FIF file's own header still has it, so copy it back. No file is changed.
header = mne.io.read_info(meg_file, verbose=False)     # header only: quick, no data
for ch_in_file in header["chs"]:
    ch = raw.info["chs"][raw.ch_names.index(ch_in_file["ch_name"])]
    if ch["kind"] == ch_in_file["kind"]:
        ch["coil_type"] = ch_in_file["coil_type"]

types = dict(zip(raw.ch_names, raw.get_channel_types()))   # errors if the repair didn't work
print("Eye-tracker channels:", {n: t for n, t in types.items() if t in ("eyegaze", "pupil")})




NameError: name 'Path' is not defined